In [1]:
import sys 
sys.path.append("/Users/shubham/Documents/Github/acceleration-flow-matching/src")


In [2]:
import torch
import torch.nn as nn
from torch.optim import Adam
from models import LatentViT
from data_loader import train_dataset
from vae import SpatialVae
from tqdm import tqdm
from flow_matching import FlowMatchingSpatialLatent
from utils import visualize_samples
from torch.utils.data import DataLoader

In [3]:
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=0)

In [4]:
def train_flowmatching(model_sampler: FlowMatchingSpatialLatent, train_loader, device='cuda', epochs=5, lr=1e-4):
    optimizer = Adam(model_sampler.model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    model_sampler.model.train()

    for epoch in range(epochs):
        total_loss = 0
        progress_bar = tqdm(train_loader, desc=f"DDPM Epoch {epoch+1}/{epochs}")

        for batch_idx, (x, y) in enumerate(progress_bar):
            
            loss, _ = model_sampler.forward_loss(x, y, criterion)

            optimizer.zero_grad()
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model_sampler.model.parameters(), 1.0)

            optimizer.step()

            total_loss += loss.item()
            progress_bar.set_postfix({'loss': loss.item()})

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1} - DDPM Loss: {avg_loss:.6f}\n")

        # if (epoch+1) % 10 == 1:
        #     torch.save({
        #         'epoch': epoch,
        #         'model_state_dict': model_sampler.model.state_dict(),
        #         'optimizer_state_dict': optimizer.state_dict(),
        #         'loss': avg_loss,
        #     }, f'{save_path}checkpoint_epoch_{epoch+1}.pth')
        #     print(f"✓ Saved checkpoint to Drive: epoch {epoch+1}, loss={avg_loss:.6f}")

    return model_sampler.model

In [5]:
device = 'cpu'
model = LatentViT(latent_channels=16, embed_dim=192, depth=12, num_heads=3)
model.load_state_dict(torch.load('/Users/shubham/Downloads/ddpm_latent_model_150.pth', map_location=torch.device(device), weights_only=True))#['model_state_dict'])
model = model.to(device)

vae = SpatialVae(in_channels=1)
vae.load_state_dict(torch.load('spatial_vae_mnist.pth', map_location=torch.device('cpu'), weights_only=True))

model_flow = FlowMatchingSpatialLatent(model, vae=vae)


In [ ]:
model = train_flowmatching(model_flow, train_loader, device=device, epochs=5, lr=1e-4)

DDPM Epoch 1/5:   0%|          | 0/235 [00:00<?, ?it/s]

DDPM Epoch 1/5:   3%|▎         | 8/235 [00:35<16:27,  4.35s/it, loss=1.86]

In [ ]:
X 